# 1. Predictions on PPMI data (m-values)

In [1]:
src_data_path = "/workspace/multi_cohort_harmonized_data.parquet"
import pandas as pd
df_multi_cohort = pd.read_parquet(src_data_path)

In [2]:
df_gse111629 = df_multi_cohort[df_multi_cohort["Cohort"] == "GSE111629"]

In [4]:
meta_cols = [col for col in df_gse111629.columns if not (col.startswith("cg") or col.startswith("ch"))]
meta_cols

['Sample_Group', 'SEX', 'Age_Group', 'Sample_Name', 'Cohort']

In [5]:
target_var = "Sample_Group"
Y_gse111629 = df_gse111629[target_var]
X_gse111629 = df_gse111629.drop(columns=meta_cols)

Y_gse111629 = Y_gse111629.map({"Control": 0, "PD": 1})

In [6]:
X_gse111629 = X_gse111629.to_numpy(dtype="float32")
Y_gse111629 = Y_gse111629.to_numpy(dtype="float32")

In [ ]:
def cv_pipeline(X_all, Y_all):

    import numpy as np
    import cupy as cp
    import cudf
    import xgboost as xgb
    from cuml.preprocessing import StandardScaler
    from cuml.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score

    print("--- Initializing 10-Fold GPU Pipeline ---")

    variances = np.var(X_all, axis=0)
    threshold = np.percentile(variances, 25) 
    high_var_mask = variances > threshold
    X_high_var = X_all[:, high_var_mask]
    print(f"Reduced features from {X_all.shape[1]} to {X_high_var.shape[1]} based on variance.")

    # Assuming X_mvalues is your pandas/cudf dataframe of harmonized M-values
    # Assuming Y_labels is your pandas/cudf series of PD_Status (0 = Control, 1 = PD)

    # Convert data to CuPy arrays to ensure it stays in GPU memory (VRAM)
    X_gpu = cp.array(X_high_var, dtype=cp.float32)
    Y_gpu = cp.array(Y_all, dtype=cp.float32)

    # Move Y to CPU just for the StratifiedKFold splitter (sklearn requires CPU labels)
    Y_cpu = cp.asnumpy(Y_gpu)

    # Initialize the 10-Fold Stratified Splitter
    # Stratified ensures the ratio of PD to Controls stays exactly the same in every fold
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    # Metrics tracking
    fold_aucs = []
    fold_pr_aucs = []
    selected_feature_counts = []

    fold = 1
    for train_idx, val_idx in skf.split(np.zeros(len(Y_cpu)), Y_cpu):
        print(f"\n--- Processing Fold {fold}/10 ---")
        
        # 1. SPLIT DATA (Still on GPU)
        X_train, X_val = X_gpu[train_idx], X_gpu[val_idx]
        Y_train, Y_val = Y_gpu[train_idx], Y_gpu[val_idx]
        
        # 2. SCALING (Fitted ONLY on training data)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val) # Transform validation data using train math
        
        # 3. FEATURE SELECTION: Elastic Net via Logistic Regression
        # l1_ratio: 0 = L2 (Ridge), 1 = L1 (Lasso), 0.5 = pure Elastic Net
        # solver='qn' (Quasi-Newton) is required for elasticnet penalty in cuml
        enet_selector = LogisticRegression(
            penalty='elasticnet', 
            l1_ratio=0.5, 
            C=0.1, # Inverse of regularization strength; tweak this to select more/fewer features
            solver='qn', 
            max_iter=1000
        )
        enet_selector.fit(X_train_scaled, Y_train)
        
        # Extract coefficients and create a boolean mask for non-zero features
        coefficients = enet_selector.coef_
        
        # cuml coef_ shape can be (1, n_features) or (n_features,). Ensure flat.
        if len(coefficients.shape) > 1:
            coefficients = coefficients[0]
            
        feature_mask = (cp.abs(coefficients) > 0)
        num_selected = int(cp.sum(feature_mask))
        selected_feature_counts.append(num_selected)
        print(f"Features selected by Elastic Net: {num_selected}")
        
        # Fallback: If Elastic Net shrinks everything to 0, use all features to prevent crash
        if num_selected == 0:
            print("Warning: Elastic Net eliminated all features. Bypassing selection for this fold.")
            feature_mask = cp.ones(X_train_scaled.shape[1], dtype=bool)
            
        # Apply the mask to both Train and Val
        X_train_selected = X_train_scaled[:, feature_mask]
        X_val_selected = X_val_scaled[:, feature_mask]
        
        # 4. XGBOOST TRAINING
        # tree_method='hist' and device='cuda' force XGBoost to run blazingly fast on the GPU
        xgb_model = xgb.XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method='hist',
            device='cuda',
            eval_metric='logloss',
            random_state=42
        )
        
        # XGBoost handles cupy arrays natively in modern versions
        xgb_model.fit(X_train_selected, Y_train)
        
        # 5. PREDICTION & METRICS
        # Predict probabilities for the validation fold
        val_probs = xgb_model.predict_proba(X_val_selected)[:, 1]
        
        # Move metrics to CPU for sklearn metric calculation
        Y_val_cpu = cp.asnumpy(Y_val)
        val_probs_cpu = cp.asnumpy(val_probs)
        
        auc = roc_auc_score(Y_val_cpu, val_probs_cpu)
        pr_auc = average_precision_score(Y_val_cpu, val_probs_cpu)
        
        fold_aucs.append(auc)
        fold_pr_aucs.append(pr_auc)
        
        print(f"Fold {fold} ROC-AUC: {auc:.4f} | PR-AUC: {pr_auc:.4f}")
        fold += 1

    # --- Final Results ---
    print("\n================ 10-FOLD CV RESULTS ================")
    print(f"Mean ROC-AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
    print(f"Mean PR-AUC:  {np.mean(fold_pr_aucs):.4f} ± {np.std(fold_pr_aucs):.4f}")
    print(f"Average Features Selected per fold: {np.mean(selected_feature_counts):.1f}")

In [ ]:
cv_pipeline(X_gse111629, Y_gse111629)

--- Initializing 10-Fold GPU Pipeline ---


Reduced features from 388296 to 97074 based on variance.

--- Processing Fold 1/10 ---
